# Project 17 · PCE Task 12개 다운로드

Task ID: 964, 966, 968, 970, 971, 972, 973, 974, 975, 976, 977, 979.

이 노트북은 `D:/Code/inmold_sw/data/PCE/task_<ID>/`에 등록 이미지, 최신 완료 Exposure annotation, alpha/이진 마스크, 오버레이, 메타데이터를 저장합니다. 원본 [`D:/Code/vision_data/temp.ipynb`](D:/Code/vision_data/temp.ipynb)의 다운로드 로직을 사용하며 Project 13 Gold 데이터는 수정하지 않습니다.

`requests`, `numpy`, `Pillow`가 필요합니다. 현재 프로젝트 가상환경으로 실행할 경우 PowerShell에서 `uv pip install --python .venv/Scripts/python.exe requests pillow ipykernel`을 한 번 실행하고 해당 커널을 선택하세요. 인증은 `LABEL_STUDIO_REFRESH_TOKEN` 환경변수 또는 인접한 `vision_data/config/label_studio_project9.local.json`을 사용합니다.

위에서 아래로 실행하면 12개를 한 장씩 저장하고 `data/PCE/download_summary.json`에 성공·실패를 기록합니다. 재실행 시 동일 Task 폴더를 최신 Annotation으로 갱신합니다.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import parse_qs, unquote, urljoin, urlsplit
from getpass import getpass
import hashlib
import io
import json
import os
import sys

import numpy as np
import requests
from PIL import Image

candidates = [Path.cwd(), *Path.cwd().parents, Path("D:/Code/inmold_sw")]
ROOT = next((p.resolve() for p in candidates if (p / "main.py").is_file() and (p / "src/PCE").is_dir()), None)
if ROOT is None:
    raise RuntimeError("inmold_sw 저장소 안에서 노트북을 열어주세요.")

PROJECT_ID = 17
VIEW_ID = None
TASK_IDS = (964, 966, 968, 970, 971, 972, 973, 974, 975, 976, 977, 979)
CONFIG_PATH = ROOT.parent / "vision_data/config/label_studio_project9.local.json"
settings = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.is_file() else {}
BASE_URL = os.environ.get("LABEL_STUDIO_URL", settings.get("url", "http://203.255.15.92:48000/")).rstrip("/") + "/"
REFRESH_TOKEN = os.environ.get("LABEL_STUDIO_REFRESH_TOKEN") or settings.get("refresh_token")
if not REFRESH_TOKEN:
    REFRESH_TOKEN = getpass("Label Studio refresh token: ")
PROJECT_URL = urljoin(BASE_URL, f"projects/{PROJECT_ID}/data")
LABEL_NAME = "Exposure"
CACHE_DIR = ROOT / "data/PCE"
DEFAULT_OPACITY = 0.45
print(f"Project {PROJECT_ID} · Task {TASK_IDS}")
print(f"저장 위치: {CACHE_DIR}")
# 토큰은 파일이나 출력에 저장하지 않습니다.


Project 17 · Task (964, 966, 968, 970, 971, 972, 973, 974, 975, 976, 977, 979)
저장 위치: D:\Code\inmold_sw\data\PCE


In [2]:
class LabelStudioClient:
    def __init__(self, base_url, refresh_token, project_id, session=None):
        self.base_url = base_url.rstrip("/") + "/"
        self.project_id = int(project_id)
        self._refresh_token = refresh_token
        self._access_token = None
        self._session = session or requests.Session()

    def _url(self, reference):
        url = urljoin(self.base_url, reference)
        wanted, actual = urlsplit(self.base_url), urlsplit(url)
        if actual.scheme not in ("http", "https") or actual.username or actual.password:
            raise ValueError("지원하지 않는 이미지/API URL입니다.")
        def origin(parts):
            return parts.scheme.lower(), parts.hostname, parts.port or (443 if parts.scheme == "https" else 80)
        if origin(wanted) != origin(actual):
            raise ValueError("다른 서버로 토큰을 전송하지 않습니다. data.image 주소를 확인하세요.")
        return url

    def _request(self, method, reference, **kwargs):
        try:
            response = self._session.request(
                method, self._url(reference), timeout=(10, 90),
                allow_redirects=False, **kwargs
            )
        except requests.RequestException:
            raise RuntimeError("Label Studio 요청 실패: 서버 연결·네트워크 상태를 확인하세요.") from None
        if 300 <= response.status_code < 400:
            response.close()
            raise RuntimeError("서버가 다른 주소로 이동시켰습니다. 서버 URL 또는 이미지 접근 권한을 확인하세요.")
        return response

    def _json(self, response):
        try:
            return response.json()
        except ValueError:
            raise RuntimeError("서버 응답이 JSON 형식이 아닙니다. URL과 접근 권한을 확인하세요.") from None
        finally:
            response.close()

    def refresh_access(self):
        response = self._request(
            "POST", "api/token/refresh/",
            json={"refresh": self._refresh_token},
            headers={"Accept": "application/json"}
        )
        if response.status_code != 200:
            status = response.status_code
            response.close()
            raise RuntimeError(f"토큰 갱신 실패 (HTTP {status}). 로컬 설정의 토큰을 확인하세요.")
        token = self._json(response).get("access")
        if not isinstance(token, str) or not token:
            raise RuntimeError("토큰 갱신 응답에 access 값이 없습니다.")
        self._access_token = token

    def get(self, reference, **kwargs):
        if self._access_token is None:
            self.refresh_access()
        for attempt in range(2):
            response = self._request(
                "GET", reference,
                headers={"Authorization": f"Bearer {self._access_token}"},
                **kwargs
            )
            if response.status_code == 401 and attempt == 0:
                response.close()
                self.refresh_access()
                continue
            if response.status_code != 200:
                status = response.status_code
                response.close()
                raise RuntimeError(f"조회 실패 (HTTP {status}). 프로젝트/파일 권한을 확인하세요.")
            return response
        raise RuntimeError("인증 갱신 후에도 요청하지 못했습니다.")

    def get_json(self, reference, **kwargs):
        return self._json(self.get(reference, **kwargs))

    def view(self, view_id):
        view = self.get_json(f"api/dm/views/{int(view_id)}/")
        project = view.get("project")
        if isinstance(project, dict):
            project = project.get("id")
        if project is None or int(project) != self.project_id or int(view["id"]) != int(view_id):
            raise RuntimeError("지정한 tab이 현재 프로젝트에 속하지 않습니다.")
        return view

    def list_task_ids(self, page_size=100, view_id=None):
        task_ids, seen, page = [], set(), 1
        while True:
            params = {"project": self.project_id, "page": page, "page_size": page_size,
                      "fields": "task_only", "query": json.dumps({"ordering": ["id"]})}
            if view_id is not None:
                params["view"] = int(view_id)
            payload = self.get_json("api/tasks/", params=params)
            if isinstance(payload, list):
                batch, total = payload, None
            elif isinstance(payload, dict):
                batch = payload.get("tasks", payload.get("results"))
                total = payload.get("total", payload.get("count"))
            else:
                raise RuntimeError("예상과 다른 작업 목록 응답입니다.")
            if not isinstance(batch, list):
                raise RuntimeError("예상과 다른 작업 목록 응답입니다.")
            if not batch:
                if total is not None and len(task_ids) < int(total):
                    raise RuntimeError("작업 목록 조회 중 개수가 바뀌었거나 페이지가 누락됐습니다. 목록을 다시 받으세요.")
                break
            for task in batch:
                task_id = int(task["id"])
                if task_id in seen:
                    raise RuntimeError("페이지에 중복 Task ID가 있습니다. 작업 목록을 다시 불러오세요.")
                seen.add(task_id)
                task_ids.append(task_id)
            if total is not None:
                if len(task_ids) >= int(total):
                    break
            elif len(batch) < page_size:
                break
            page += 1
        return sorted(task_ids)

    def task(self, task_id):
        task = self.get_json(f"api/tasks/{int(task_id)}/")
        project = task.get("project")
        if isinstance(project, dict):
            project = project.get("id")
        if project is None or int(project) != self.project_id or int(task["id"]) != int(task_id):
            raise RuntimeError("Task ID 또는 프로젝트가 현재 설정과 다릅니다.")
        return task

    def image_bytes(self, reference):
        response = self.get(reference)
        try:
            return response.content
        finally:
            response.close()

    def close(self):
        self._session.close()



In [3]:
def decode_brush_alpha(result: dict) -> np.ndarray:
    """Decode the RLE format used by this project's Label Studio brush exports.

    Preserve intermediate alpha values; binary membership is alpha > 0.
    Reject rotated, truncated or incorrectly sized masks instead of misaligning
    their coordinates with source images.
    """
    width, height = result['original_width'], result['original_height']
    if (type(width) is not int or type(height) is not int or width <= 0 or height <= 0):
        raise ValueError('Brush dimensions must be positive integers')
    if result.get('image_rotation', 0) != 0:
        raise ValueError('Rotated brush results are unsupported')
    if result['value']['format'] != 'rle':
        raise ValueError('Expected brush RLE format')
    packed = bytes(result['value']['rle'])
    bits = ''.join(format(v, '08b') for v in packed)
    cursor = 0

    def read(count: int) -> int:
        nonlocal cursor
        end = cursor + count
        if end > len(bits):
            raise ValueError('Truncated brush RLE input')
        value = int(bits[cursor:end], 2)
        cursor = end
        return value

    count, word_size = read(32), read(5) + 1
    lengths = [read(4) + 1 for _ in range(4)]
    if count != width * height * 4 or word_size != 8:
        raise ValueError('Brush RLE dimensions or word size do not match RGBA')
    output = np.empty(count, dtype=np.uint8)
    index = 0
    while index < count:
        repeated = read(1)
        length = read(lengths[read(2)]) + 1
        end = index + length
        if end > count:
            raise ValueError('Brush RLE run exceeds image dimensions')
        if repeated:
            output[index:end] = read(word_size)
        else:
            for i in range(index, end):
                output[i] = read(word_size)
        index = end
    return output.reshape(height, width, 4)[:, :, 3].copy()


In [4]:
def select_latest_annotation(task):
    annotations = [a for a in task.get("annotations", []) if not a.get("was_cancelled", False)]
    if not annotations:
        raise ValueError(f"Task {task['id']}: 완료된 Annotation이 없습니다.")
    def order(a):
        timestamp = a.get("updated_at") or a.get("created_at")
        if not timestamp:
            raise ValueError("Annotation 수정 시간이 없어 최신 버전을 결정할 수 없습니다.")
        when = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        if when.tzinfo is None:
            when = when.replace(tzinfo=timezone.utc)
        return when, int(a["id"])
    return max(annotations, key=order)



def annotation_alpha(annotation, width, height, label="Exposure"):
    alpha = np.zeros((height, width), dtype=np.uint8)
    regions = [r for r in annotation.get("result", [])
               if r.get("type") == "brushlabels" and label in r.get("value", {}).get("brushlabels", [])]
    for region in regions:
        decoded = decode_brush_alpha(region)
        if decoded.shape != alpha.shape:
            raise ValueError(f"등록 이미지 {width}×{height}와 마스크 {decoded.shape[::-1]}의 크기가 다릅니다.")
        alpha = np.maximum(alpha, decoded)
    if not regions and annotation.get("result"):
        raise ValueError(f"최신 Annotation에 {label!r} brushlabels가 없습니다.")
    # 비어 있는 완료 Annotation은 명시적인 빈 마스크로 표시합니다.
    return alpha



def overlay_pixels(rgb, mask, opacity=0.45, color=(255, 70, 70)):
    if rgb.shape[:2] != mask.shape or not 0 <= opacity <= 1:
        raise ValueError("오버레이 크기 또는 투명도를 확인하세요.")
    result = rgb.copy()
    result[mask] = np.rint(
        (1 - opacity) * rgb[mask].astype(np.float32)
        + opacity * np.asarray(color, dtype=np.float32)
    ).astype(np.uint8)
    return result



def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)



def write_bytes(path, blob):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(blob)
    temporary.replace(path)


def write_png(path, pixels):
    temporary = path.with_suffix(path.suffix + ".tmp")
    Image.fromarray(pixels).save(temporary, format="PNG")
    temporary.replace(path)


def write_numpy(path, pixels):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as stream:
        np.save(stream, pixels, allow_pickle=False)
    temporary.replace(path)


def download_one(client, task_id, output_root=CACHE_DIR, label=LABEL_NAME):
    task = client.task(task_id)
    annotation = select_latest_annotation(task)
    reference = task.get("data", {}).get("image")
    if not isinstance(reference, str) or not reference:
        raise ValueError(f"Task {task_id}: data.image가 없습니다.")
    blob = client.image_bytes(reference)
    try:
        with Image.open(io.BytesIO(blob)) as loaded:
            loaded.load()
            image_format = loaded.format
            rgb = np.array(loaded.convert("RGB"))
    except (OSError, ValueError):
        raise ValueError("다운로드 응답을 이미지로 읽지 못했습니다. 이미지 접근 권한을 확인하세요.") from None
    height, width = rgb.shape[:2]
    alpha = annotation_alpha(annotation, width, height, label)
    mask = alpha > 0
    folder = Path(output_root) / f"task_{int(task_id)}"
    previous_metadata = folder / "metadata.json"
    if previous_metadata.is_file():
        previous = json.loads(previous_metadata.read_text(encoding="utf-8"))
        if int(previous["project_id"]) != int(client.project_id):
            raise ValueError("같은 Task 폴더에 다른 프로젝트의 파일이 있습니다. 별도 저장 폴더를 지정하세요.")
    folder.mkdir(parents=True, exist_ok=True)
    if image_format == "PNG":
        write_bytes(folder / "image.png", blob)
    else:
        write_png(folder / "image.png", rgb)
        write_bytes(folder / "source_image.bin", blob)
    write_png(folder / "mask_alpha.png", alpha)
    write_png(folder / "mask.png", mask.astype(np.uint8) * 255)
    write_numpy(folder / "mask_alpha.npy", alpha)
    write_numpy(folder / "mask.npy", mask.astype(np.uint8))
    write_png(folder / "overlay.png", overlay_pixels(rgb, mask, DEFAULT_OPACITY))
    selected_task = {"id": int(task_id), "project": int(client.project_id),
                     "data": task["data"], "annotations": [annotation]}
    write_json(folder / "annotation.json", selected_task)
    write_json(folder / "task.json", task)
    source_path = parse_qs(urlsplit(reference).query).get("d", [unquote(urlsplit(reference).path)])[0]
    filenames = ("image.png", "mask_alpha.png", "mask.png", "mask_alpha.npy", "mask.npy",
                 "overlay.png", "annotation.json", "task.json")
    metadata = {
        "status": "completed", "task_id": int(task_id), "project_id": int(client.project_id),
        "annotation_id": int(annotation["id"]), "annotation_updated_at": annotation.get("updated_at"),
        "image_reference": reference, "source_filename": Path(source_path).name,
        "width": width, "height": height, "image_source": "label_studio_registered",
        "label": label, "mask_rule": "alpha > 0", "mask_png_values": [0, 255], "mask_npy_values": [0, 1],
        "exposure_pixels": int(mask.sum()), "empty_completed_annotation": not bool(annotation.get("result")),
        "download_sha256": hashlib.sha256(blob).hexdigest(),
        "image_sha256": hashlib.sha256((folder / "image.png").read_bytes()).hexdigest(),
        "annotation_sha256": hashlib.sha256(json.dumps(annotation, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest(),
        "files_sha256": {name: hashlib.sha256((folder / name).read_bytes()).hexdigest() for name in filenames},
        "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(folder / "metadata.json", metadata)
    return {"rgb": rgb, "alpha": alpha, "mask": mask, "folder": folder, "metadata": metadata}


In [5]:
client = LabelStudioClient(BASE_URL, REFRESH_TOKEN, PROJECT_ID)
project = client.get_json(f"api/projects/{PROJECT_ID}/")
if int(project["id"]) != PROJECT_ID:
    raise RuntimeError("프로젝트 응답 ID가 설정과 다릅니다.")
if len(set(TASK_IDS)) != len(TASK_IDS):
    raise ValueError("중복 Task ID가 있습니다.")
DOWNLOAD_TASK_IDS = tuple(TASK_IDS)
for task_id in DOWNLOAD_TASK_IDS:
    client.task(task_id)  # 저장 전에 12개 모두 Project 17 소속인지 검증합니다.
print(f"검증 완료: {project.get('title', '')} · {len(DOWNLOAD_TASK_IDS)}개")


검증 완료: 2026.10.26_검증용 · 12개


In [6]:
RUN_DOWNLOAD_ALL = True  # False이면 자동 저장하지 않고 아래 선택 셀만 사용할 수 있습니다.
if RUN_DOWNLOAD_ALL:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    downloaded_records = {}  # 작은 metadata만 보관. 이미지/마스크 배열은 보관하지 않습니다.
    download_errors = {}
    def save_download_summary():
        write_json(CACHE_DIR / "download_summary.json", {
            "project_id": PROJECT_ID, "view_id": VIEW_ID, "project_url": PROJECT_URL,
            "requested_task_ids": list(DOWNLOAD_TASK_IDS), "saved_count": len(downloaded_records),
            "completed": len(downloaded_records) == len(DOWNLOAD_TASK_IDS),
            "tasks": list(downloaded_records.values()), "errors": download_errors,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        })
    save_download_summary()
    for index, task_id in enumerate(DOWNLOAD_TASK_IDS, 1):
        record = None
        print(f"[{index}/{len(DOWNLOAD_TASK_IDS)}] Project {PROJECT_ID} · Task {task_id}: 다운로드 중...", flush=True)
        try:
            record = download_one(client, task_id)
            metadata = record["metadata"]
            downloaded_records[task_id] = metadata
            print(f"완료: Task {task_id} | Annotation {metadata['annotation_id']} | "
                  f"{metadata['width']}×{metadata['height']} | Exposure {metadata['exposure_pixels']:,} px | "
                  f"저장: {record['folder']}", flush=True)
        except Exception as error:
            download_errors[task_id] = f"{type(error).__name__}: {error}"
            print(f"실패: Task {task_id} · {download_errors[task_id]}", flush=True)
        finally:
            record = None
            save_download_summary()
    print(f"저장 완료: {len(downloaded_records)}/{len(DOWNLOAD_TASK_IDS)}개 · {CACHE_DIR}")
    if download_errors:
        raise RuntimeError(f"실패한 Task ID: {sorted(download_errors)}. 위 오류와 download_summary.json을 확인하세요.")
else:
    print("전체 다운로드는 꺼져 있습니다. 아래 한 장/한 장씩 확인 셀을 사용하세요.")


[1/12] Project 17 · Task 964: 다운로드 중...
완료: Task 964 | Annotation 626 | 2046×2046 | Exposure 196,845 px | 저장: D:\Code\inmold_sw\data\PCE\task_964
[2/12] Project 17 · Task 966: 다운로드 중...
완료: Task 966 | Annotation 628 | 2046×2046 | Exposure 1,886,618 px | 저장: D:\Code\inmold_sw\data\PCE\task_966
[3/12] Project 17 · Task 968: 다운로드 중...
완료: Task 968 | Annotation 630 | 2046×2046 | Exposure 185,335 px | 저장: D:\Code\inmold_sw\data\PCE\task_968
[4/12] Project 17 · Task 970: 다운로드 중...
완료: Task 970 | Annotation 632 | 2046×2046 | Exposure 200,186 px | 저장: D:\Code\inmold_sw\data\PCE\task_970
[5/12] Project 17 · Task 971: 다운로드 중...
완료: Task 971 | Annotation 633 | 2046×2046 | Exposure 1,899,259 px | 저장: D:\Code\inmold_sw\data\PCE\task_971
[6/12] Project 17 · Task 972: 다운로드 중...
완료: Task 972 | Annotation 634 | 2046×2046 | Exposure 196,640 px | 저장: D:\Code\inmold_sw\data\PCE\task_972
[7/12] Project 17 · Task 973: 다운로드 중...
완료: Task 973 | Annotation 635 | 2046×2046 | Exposure 205,017 px | 저장: D:\Code\in